# Árvores geradoras na matriz P completa

Experimento independente com as 67 atividades, sem “Outras”. Comparamos a árvore mínima bilateral, a máxima bilateral e a arborescência máxima dirigida. A matriz P atribui emissões da atividade emissora (linha i) à demanda final pelos produtos da atividade j (coluna).

As definições econômicas estão no [README](README.md#nota-metodológica). Esta exploração não altera as métricas ou as figuras anteriores.

In [1]:
from pathlib import Path
import json
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from redes import dados
from redes.redes import validar_matriz, carregar_setores, matriz_para_grafo
from redes.visualizacoes_arvores import figura_arvore, exportar_pagina_arvores

RAIZ = dados.RAIZ_PROJETO


## 1. Entrada experimental e proveniência

O CSV local de P diverge do hash do manifesto canônico. Foi preservada uma cópia byte a byte para este experimento, com hash próprio registrado. Isso permite repetir esta exploração, mas não certifica a equivalência da entrada ao manifesto original. A checagem canônica permanece intacta; não regeneramos P neste notebook.

In [2]:
# Origem: cópia experimental do CSV exportado pela etapa MIP; não abre planilhas.
proveniencia = json.loads((RAIZ / "raw/experimento_arvores/proveniencia.json").read_text(encoding="utf-8"))
caminho = RAIZ / proveniencia["copia"]
dados.check_sha256(caminho, proveniencia["sha256_copia"])
P = pd.read_csv(caminho, dtype={"atividade_emissora": str},
                index_col="atividade_emissora", float_precision="round_trip")
validar_matriz(P)
assert P.shape == (67, 67)
P_original = P.copy(deep=True)
setores = carregar_setores("setores_mip_2015")
assert set(setores.index) == set(P.index)
setores = setores.reindex(P.index)
# Dimensões: 67 × 67; unidade: Gg de CO₂; linhas emissoras e colunas destinos.
display(pd.Series(proveniencia, name="Proveniência"))


SHA-256 verificado: P.csv
SHA-256 verificado: setores_2015.csv


origem                               outputs/matriz_emissoes_producao_2015.csv
copia                                            raw/experimento_arvores/P.csv
sha256_copia                 47c25e57c53c5b59503d7381ef0c462a9b7f1141d4bf96...
sha256_manifesto_canonico    99DF37C9CCE571783DB0D54280766F53F4E6FBB577E1EA...
nota                         Cópia experimental autorizada do CSV local; di...
Name: Proveniência, dtype: str

## 2. Da matriz aos grafos

Retiramos a diagonal somente da seleção de ligações: $W_{ij}=P_{ij}$ para $i\ne j$ e $W_{ii}=0$. O grafo dirigido preserva $i\to j$. Para as árvores bilaterais, usamos $S=W+W^T$: cada par não dirigido tem peso $P_{ij}+P_{ji}$, contado uma única vez.

Zeros representam ausência de relação. Atividades isoladas permanecem no conjunto de nós.

In [3]:
# Origem: P completa; dimensões e unidade permanecem iguais.
valores = P.to_numpy(copy=True)
np.fill_diagonal(valores, 0)
W = pd.DataFrame(valores, index=P.index, columns=P.columns)
S = W + W.T
G = matriz_para_grafo(W)
B = nx.from_pandas_adjacency(S, create_using=nx.Graph)
assert nx.number_of_selfloops(G) == nx.number_of_selfloops(B) == 0
np.testing.assert_allclose(B.size(weight="weight"), W.to_numpy().sum())
componentes = list(nx.weakly_connected_components(G))
isolados = list(nx.isolates(G))
print("Atividades:", len(G), "| Ligações dirigidas:", G.number_of_edges())
print("Tamanhos dos componentes:", [len(c) for c in componentes])
print("Isoladas:", [(i, setores[i]) for i in isolados])


Atividades: 67 | Ligações dirigidas: 4290
Tamanhos dos componentes: [66, 1]
Isoladas: [('9700', 'Serviços domésticos')]


## 3. Seleção das ligações

A árvore mínima minimiza a soma dos pesos; a máxima a maximiza, mantendo a conexão sem ciclos. Como há um nó isolado, obtemos florestas: uma árvore com 66 atividades e um nó separado.

A arborescência máxima preserva a direção e admite uma única entrada por atividade conectada, exceto na raiz. A raiz é escolhida pela otimização, sem imposição econômica. Ela não identifica a origem causal da produção. Se um componente não admitir arborescência, a execução informa o problema, sem substituição por outro método.

In [4]:
# NetworkX seleciona as ligações; os pesos originais permanecem em weight.
arvore_minima = nx.minimum_spanning_tree(B, weight="weight", algorithm="kruskal")
arvore_maxima = nx.maximum_spanning_tree(B, weight="weight", algorithm="kruskal")
arvore_dirigida = nx.DiGraph()
arvore_dirigida.add_nodes_from(G.nodes)
raizes = []
for componente in componentes:
    if len(componente) == 1:
        continue
    subgrafo = G.subgraph([i for i in G if i in componente]).copy()
    try:
        arvore = nx.maximum_spanning_arborescence(subgrafo, attr="weight", preserve_attrs=True)
    except nx.NetworkXException as erro:
        raise ValueError("Um componente de P não admite arborescência geradora; não foi aplicado método alternativo.") from erro
    # Edmonds faz subtrações internas: recuperar o peso exato de cada ligação em G.
    for i, j in arvore.edges:
        np.testing.assert_allclose(arvore[i][j]["weight"], G[i][j]["weight"], rtol=1e-10, atol=1e-10)
        arvore[i][j]["weight"] = G[i][j]["weight"]
    arvore_dirigida.add_edges_from(arvore.edges(data=True))
    raizes.extend(i for i, grau in arvore.in_degree() if grau == 0)
resultados = {"minima": arvore_minima, "maxima": arvore_maxima, "dirigida": arvore_dirigida}
print("Raízes:", [(i, setores[i]) for i in raizes])


Raízes: [('8000', 'Atividades de vigilância, segurança e investigação')]


## 4. Comparação e indicadores do desenho

O denominador da cobertura é $\sum_{i\ne j}P_{ij}$. Nas árvores bilaterais, somamos os pares selecionados uma vez; na dirigida, apenas as setas selecionadas. Cobertura de peso não implica equivalência estrutural entre os métodos.

O tamanho dos nós continua baseado em $\sum_j P_{ij}$, incluindo a diagonal. A cor usa a diversidade intersetorial original: $D_i=1/\sum_j q_{ij}^2$, com $q_{ij}=W_{ij}/\sum_j W_{ij}$. Adotamos zero para atividades sem saídas, cuja distribuição é indefinida. Esses indicadores não são recalculados nas árvores.

In [5]:
nomes = {"minima": "Mínima bilateral", "maxima": "Máxima bilateral", "dirigida": "Máxima dirigida"}
peso_intersetorial = W.to_numpy().sum()
linhas = []
for chave, arvore in resultados.items():
    peso = arvore.size(weight="weight")
    linhas.append({"abordagem": nomes[chave], "nos": len(arvore), "ligacoes": arvore.number_of_edges(),
                   "componentes": nx.number_connected_components(arvore.to_undirected()),
                   "peso_gg": peso, "participacao_pct": 100 * peso / peso_intersetorial if peso_intersetorial > 0 else np.nan})
resumo = pd.DataFrame(linhas)
metricas = pd.DataFrame({"descricao": setores, "emissoes_proprias": P.sum(axis=1)})
q = W.div(W.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
metricas["destinos_efetivos"] = (1 / q.pow(2).sum(axis=1).replace(0, np.nan)).fillna(0)
display(resumo)


,abordagem,nos,ligacoes,componentes,peso_gg,participacao_pct
0,Mínima bilateral,67,65,2,241.295392,0.082982
1,Máxima bilateral,67,65,2,88919.634120,30.579744
2,Máxima dirigida,67,65,2,58038.817249,19.959733


## 5. Validação e interpretação

Uma árvore com n nós conectados tem n−1 ligações. Verificamos ausência de ciclos, componentes, pesos e direções. Os caminhos desenhados não representam etapas físicas adicionais de produção: P já incorpora requisitos diretos e indiretos. As relações omitidas continuam relevantes; as árvores são resumos exploratórios.

In [6]:
pd.testing.assert_frame_equal(P, P_original)
for chave, arvore in resultados.items():
    assert set(arvore) == set(P.index)
    assert nx.is_forest(arvore.to_undirected())
    assert arvore.number_of_edges() == len(P) - len(componentes)
    referencia = G if arvore.is_directed() else B
    for i, j, atributos in arvore.edges(data=True):
        assert atributos["weight"] == referencia[i][j]["weight"]
for componente in componentes:
    if len(componente) > 1:
        assert nx.is_arborescence(arvore_dirigida.subgraph(componente))
assert arvore_minima.size(weight="weight") <= arvore_maxima.size(weight="weight")
assert resumo.participacao_pct.between(0, 100).all()
print("P preservada; nós, pesos, direção e estrutura das seleções verificados.")


P preservada; nós, pesos, direção e estrutura das seleções verificados.


## 6. Figuras e publicação

Cada árvore recebe seu próprio layout Kamada–Kawai, calculado sobre as ligações selecionadas, com física desligada. Usamos distâncias topológicas (número de ligações), não volumes de emissões como comprimentos. Na arborescência, apenas o cálculo das posições usa a projeção não dirigida; as setas e os pesos são preservados. O isolado ocupa uma margem separada. Área proporcional às emissões próprias; emissão zero recebe um losango de identificação. As espessuras usam a mesma escala em Gg, e as cores a mesma escala de diversidade. Zoom e arraste permitem explorar as 67 atividades.

A execução gera exclusivamente a [página de árvores](docs/index.html), seus três desenhos e downloads. Esta é a página principal; a exploração inicial permanece em arquivo separado.

In [7]:
# Layout é apenas visual: não representa distância econômica.
max_peso = max((a["weight"] for _, _, a in B.edges(data=True)), default=0)
pasta = RAIZ / "docs/arvores"
pasta.mkdir(parents=True, exist_ok=True)
for chave, arvore in resultados.items():
    # Desenhar a topologia selecionada; pesos em Gg não são distâncias.
    conectados = [i for i in arvore if arvore.degree(i) > 0]
    estrutura = arvore.subgraph(conectados).to_undirected()
    posicoes = nx.kamada_kawai_layout(estrutura, weight=None) if conectados else {}
    for ordem, codigo in enumerate(isolados):
        posicoes[codigo] = np.array([1.3, 1.0 - ordem * 0.15])
    figura = figura_arvore(arvore, metricas, posicoes, max_peso)
    html = figura.generate_html().replace("</head>", "<style>.vis-tooltip{white-space:pre-line}</style></head>")
    (pasta / f"{chave}.html").write_text(html, encoding="utf-8")
    # Bilateral: atividade_1/2 não expressam direção. Dirigida: origem/destino expressam i → j.
    colunas = ["origem", "destino"] if arvore.is_directed() else ["atividade_1", "atividade_2"]
    registros = [{colunas[0]: i, colunas[1]: j, "descricao_1": setores[i], "descricao_2": setores[j],
                  "peso_gg_co2": a["weight"]} for i, j, a in arvore.edges(data=True)]
    pd.DataFrame(registros, columns=colunas + ["descricao_1", "descricao_2", "peso_gg_co2"]).to_csv(pasta / f"{chave}.csv", index=False)
resumo.to_csv(pasta / "comparacao.csv", index=False)
exportar_pagina_arvores(RAIZ / "docs/index.html", resultados, resumo, proveniencia,
                        [f"{i} — {setores[i]}" for i in raizes],
                        [f"{i} — {setores[i]}" for i in isolados], metricas.destinos_efetivos.max())
print("Página gerada: docs/index.html")


Página gerada: docs/index.html
